# 💰 The Political Layer — Executive Brief

**The Data Vigilante | Sierra Napier, MPA**

> *"This is not a smoking gun. It is a pattern of access."*

---

## What This Notebook Examines

Seven federal legislators with UI-relevant committee assignments were analyzed using FEC 2023-2024 reporting period data. The question is not whether money buys votes. The question is: **who is in the donor file, and who isn't?**

## Key Numbers

| Metric | Value |
|--------|-------|
| Members analyzed | 7 |
| Total committee activity | **$89.9M** |
| Excluding Trone's $62.9M in self-loans | **$26.1M** |
| Members with $0 itemized labor contributions | **4 of 7** |
| David Trone self-funding rate | **98.7%** |

## Who These Members Are

- **David Trone (MD)** — Co-founder of Total Wine & More (~$2.4B company). Sat on **Ways and Means Committee** (writes tax law for alcohol retailers) while self-funding 98.7% of his Senate campaign from Total Wine wealth.
- **Tim Kaine (VA)** — Senate HELP Committee (Health, Education, Labor, and Pensions). 2024 re-election.
- **Mark Warner (VA)** — Senate Finance + Banking committees. **NOT a 2024 candidate** (next race 2026). His $4.27M reflects off-cycle committee fundraising.
- **Steny Hoyer (MD)** — House Appropriations. 2024 re-election. Did not seek re-election in 2026.
- **Ben Cline (VA)** — House Appropriations + Budget. 2024 re-election.
- **Dutch Ruppersberger (MD)** — House Appropriations. 2024 re-election.
- **Chris Van Hollen (MD)** — Senate Appropriations + Banking + Budget. **NOT a 2024 candidate** (~2028 race). Figures reflect off-cycle activity.

## Methodology Disclosure

- Contribution categorization uses keyword pattern matching on contributor names — approximate, not based on official FEC industry codes
- "Self-funding" detection uses last-name matching; may over-count joint committee transfers for Kaine, Warner, and Hoyer
- Only David Trone is a confirmed personal self-funder (candidate loans to own committee, per FEC.gov)
- Business = corporate PACs + employer-named organizations; Labor = union PACs + labor-named organizations
- Data source: FEC API v1, cycle=2024, Schedule A, ≥$500 itemized contributions

**Sources:** [FEC Open API](https://api.open.fec.gov) · [OpenSecrets](https://www.opensecrets.org) · [FEC.gov campaign finance data](https://www.fec.gov/data/)  
**Live portfolio:** https://thedatavigilante.github.io/UI_INDEX/

# 🏛️ Political Accountability Layer

**The Data Vigilante | Sierra Napier, MPA**

Maps legislative control over UI/SUI policy to campaign finance data. Seven members of Congress from DC, Maryland, and Virginia sit on committees with jurisdiction over unemployment insurance — and their funding sources tell a story.

> *"The same people who vote on your benefit cap are funded by the industries that profit from keeping it low."*

---
**Sources:** FEC Open API (2024 cycle) · Congress.gov (119th Congress) · Census ACS 2022  
**Live portfolio:** https://thedatavigilante.github.io/UI_INDEX/

In [1]:
import json, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import pandas as pd
from pathlib import Path

BG='#121212'; BG2='#1e1e1e'; GRID='#2a2a2a'; FG='#e8e8e8'; MUTED='#888888'
GREEN='#00FF41'; LIME='#BFFF00'; GOLD='#D4AF37'; CRIMSON='#DC143C'; BLUE='#4aa8d8'; ORANGE='#F39C12'
STATE_COLORS = {'MD': BLUE, 'VA': GREEN, 'DC': GOLD}

def style_ax(ax, title, xlabel=None, ylabel=None):
    ax.set_facecolor(BG2); ax.figure.patch.set_facecolor(BG)
    ax.set_title(title, color=LIME, fontsize=12, fontweight='bold', fontfamily='monospace', pad=12)
    if xlabel: ax.set_xlabel(xlabel, color=MUTED, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=MUTED, fontsize=10)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.spines[:].set_color(GRID)
    ax.grid(color=GRID, linewidth=0.5, linestyle='--', alpha=0.6)

with open('data/political/fec_funding_profiles.json') as f:
    raw = json.load(f)
profiles = raw.get('data', raw) if isinstance(raw, dict) else raw

with open('data/political/political_layer_report.json') as f:
    report = json.load(f)

fec_df = pd.DataFrame(profiles)
print(f'FEC profiles: {len(fec_df)} members')
print(f'Total receipts: ${fec_df["total_receipts"].sum()/1e6:.1f}M')
fec_df[['name','state','total_receipts','business_contributions','labor_contributions']].sort_values('total_receipts', ascending=False)

FEC profiles: 7 members
Total receipts: $89.9M


,name,state,total_receipts,business_contributions,labor_contributions
0,David Trone,MD,63833136.59,0.00,0.0
4,Tim Kaine,VA,18308453.68,0.00,0.0
3,Mark Warner,VA,4266868.81,0.00,0.0
6,Steny Hoyer,MD,1756898.25,31669.23,0.0
2,Ben Cline,VA,1030467.03,65000.00,5000.0
5,Dutch Ruppersberger,MD,373512.72,105000.00,5000.0
1,Chris Van Hollen,MD,330578.51,57899.24,0.0


## Chart 1 — Total Campaign Receipts

Who's raising the most money? Scale matters when analyzing whose attention gets bought.

In [2]:
sorted_profiles = sorted(profiles, key=lambda p: p['total_receipts'])
fig, ax = plt.subplots(figsize=(11, 6))
style_ax(ax, 'Total Campaign Receipts — UI Committee Members (FEC, 2024 Cycle)', xlabel='Total Receipts ($)')
names  = [f"{p['name']}  ({p['state']})" for p in sorted_profiles]
totals = [p['total_receipts'] for p in sorted_profiles]
colors = [STATE_COLORS.get(p['state'], GOLD) for p in sorted_profiles]
bars = ax.barh(range(len(sorted_profiles)), totals, color=colors, edgecolor=GRID, linewidth=0.6, height=0.6)
ax.set_yticks(range(len(sorted_profiles)))
ax.set_yticklabels(names, color=FG, fontsize=9, fontfamily='monospace')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M' if x>=1e6 else f'${x/1e3:.0f}K'))
for bar, val in zip(bars, totals):
    label = f'${val/1e6:.1f}M' if val>=1e6 else f'${val/1e3:.0f}K'
    ax.text(bar.get_width()+max(totals)*0.01, bar.get_y()+bar.get_height()/2, label, va='center', ha='left', fontsize=9, color=FG, fontfamily='monospace', fontweight='bold')
legend_patches = [mpatches.Patch(color=BLUE, label='Maryland'), mpatches.Patch(color=GREEN, label='Virginia'), mpatches.Patch(color=GOLD, label='DC')]
ax.legend(handles=legend_patches, facecolor=BG2, edgecolor=GRID, labelcolor=FG, fontsize=9, loc='lower right')
ax.set_xlim(0, max(totals)*1.18)
plt.tight_layout(); plt.show()

/var/folders/ll/f8_rnhhs6sbd7zfhnvg7hbsw0000gn/T/ipykernel_42913/2375772232.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🔍 Investigative Take — Total Receipts

David Trone raised $63.8M and loaned nearly all of it to himself. That $62.9M came from wealth built co-founding Total Wine & More — the largest private wine and spirits retailer in the country, valued at roughly $2.4 billion. Remove the self-funder, and the remaining six members raised a combined $26.1M.

These are the legislators in the rooms where UI benefit caps are set, left unchanged, and defended. $26M buys a lot of rooms. Ways and Means — Trone's committee — writes the tax code that governs alcohol retailers. The founder of the country's largest alcohol retailer sat on that committee while funding his campaign from that wealth.

> *"The FEC calls it 'candidate loans.' The math is the same either way."*


## Chart 2 — Business vs. Labor Contributions

Of the itemized contributions (≥$500), how much comes from business/industry vs. labor unions? This is where the alignment of interests becomes visible.

In [3]:
fig, ax = plt.subplots(figsize=(11, 6))
style_ax(ax, 'Business vs. Labor Contributions — UI Committee Members (FEC Schedule A, ≥$500)', ylabel='Itemized Contributions ($)')
x = range(len(profiles)); width = 0.55
biz   = [p.get('business_contributions',0) for p in profiles]
labor = [p.get('labor_contributions',0) for p in profiles]
other = [p.get('other_contributions',0) for p in profiles]
ax.bar(x, biz,   width, label='Business / Industry', color=CRIMSON, edgecolor=GRID, linewidth=0.6)
ax.bar(x, labor, width, bottom=biz, label='Labor Unions', color=BLUE, edgecolor=GRID, linewidth=0.6)
ax.bar(x, other, width, bottom=[b+l for b,l in zip(biz,labor)], label='Other / PAC', color=GOLD, edgecolor=GRID, linewidth=0.6, alpha=0.85)
xlabels = [f"{p['name']}\n({p['state']})" for p in profiles]
ax.set_xticks(list(x)); ax.set_xticklabels(xlabels, color=FG, fontsize=8, fontfamily='monospace', rotation=10, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y,_: f'${y/1e3:.0f}K' if y>0 else '$0'))
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=FG, fontsize=9)
for i,p in enumerate(profiles):
    total_i = p.get('business_contributions',0)+p.get('labor_contributions',0)+p.get('other_contributions',0)
    if total_i>0:
        ax.text(i, total_i+max([b+l+o for b,l,o in zip(biz,labor,other)])*0.02, f'${total_i/1e3:.0f}K', ha='center', va='bottom', fontsize=8, color=MUTED, fontfamily='monospace')
plt.tight_layout(); plt.show()

/var/folders/ll/f8_rnhhs6sbd7zfhnvg7hbsw0000gn/T/ipykernel_42913/261021013.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [4]:
# ── BUSINESS-TO-LABOR RATIO ANALYSIS ────────────────────────────────────────
import json

with open('data/political/fec_funding_profiles.json') as f:
    profiles = json.load(f)['data']

print('=== BUSINESS-TO-LABOR RATIO — UI Committee Members ===\n')
print(f"{'Member':<22} {'State':>5} {'Business':>12} {'Labor':>10} {'B:L Ratio':>12}  Notes")
print('─' * 80)

ratios = []
for p in profiles:
    biz  = p.get('business_contributions', 0)
    lab  = p.get('labor_contributions', 0)
    name = p['name']
    state = p['state']
    ratio_str = f'{biz/lab:.1f}:1' if lab > 0 else '∞  (zero labor)'
    ratio_val = biz/lab if lab > 0 else float('inf')
    note = ''
    if 'Trone' in name:    note = '← 98.7% self-funded; Ways & Means + Total Wine'
    if 'Warner' in name:   note = '← 2026 candidate; off-cycle'
    if 'Van Hollen' in name: note = '← ~2028 candidate; off-cycle'
    print(f'{name:<22} {state:>5} ${biz:>11,.0f} ${lab:>9,.0f} {ratio_str:>12}  {note}')
    ratios.append((ratio_val, name, biz, lab))

print()
print('KEY FINDING: 4 of 7 members show $0 in itemized labor contributions (≥$500 threshold).')
print('The workers whose benefit adequacy these members govern are not in the donor file.')


=== BUSINESS-TO-LABOR RATIO — UI Committee Members ===

Member                 State     Business      Labor    B:L Ratio  Notes
────────────────────────────────────────────────────────────────────────────────
David Trone               MD $          0 $        0 ∞  (zero labor)  ← 98.7% self-funded; Ways & Means + Total Wine
Chris Van Hollen          MD $     57,899 $        0 ∞  (zero labor)  ← ~2028 candidate; off-cycle
Ben Cline                 VA $     65,000 $    5,000       13.0:1  
Mark Warner               VA $          0 $        0 ∞  (zero labor)  ← 2026 candidate; off-cycle
Tim Kaine                 VA $          0 $        0 ∞  (zero labor)  
Dutch Ruppersberger       MD $    105,000 $    5,000       21.0:1  
Steny Hoyer               MD $     31,669 $        0 ∞  (zero labor)  

KEY FINDING: 4 of 7 members show $0 in itemized labor contributions (≥$500 threshold).
The workers whose benefit adequacy these members govern are not in the donor file.


In [5]:
# Business vs. Labor bar chart (branded)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

BG='#121212'; BG2='#1e1e1e'; GRID='#2a2a2a'; FG='#e8e8e8'; MUTED='#888888'
GREEN='#00FF41'; LIME='#BFFF00'; GOLD='#D4AF37'; CRIMSON='#DC143C'

names = [p['name'].split()[-1] for p in profiles]  # last names for x-axis
biz_vals = [p.get('business_contributions',0)/1000 for p in profiles]
lab_vals  = [p.get('labor_contributions',0)/1000 for p in profiles]
x = np.arange(len(names))
w = 0.35

fig, ax = plt.subplots(figsize=(11,6))
fig.patch.set_facecolor(BG); ax.set_facecolor(BG2)
ax.bar(x-w/2, biz_vals, w, color=CRIMSON, label='Business contributions', alpha=0.9)
ax.bar(x+w/2, lab_vals, w, color=GREEN, label='Labor contributions', alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(names, color=FG, fontsize=9, fontfamily='monospace')
ax.set_ylabel('Amount ($K)', color=MUTED, fontsize=10)
ax.set_title('Business vs. Labor Contributions — UI Committee Members (FEC 2024 cycle)',
             color=LIME, fontsize=12, fontweight='bold', fontfamily='monospace', pad=12)
ax.tick_params(colors=MUTED, labelsize=9)
ax.spines[:].set_color(GRID)
ax.grid(color=GRID, linewidth=0.6, linestyle='--', alpha=0.7, axis='y')
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=FG, fontsize=9)
ax.annotate('4 of 7 members: $0 labor at ≥$500 threshold', xy=(0.5,0.92),
            xycoords='axes fraction', ha='center', color=GOLD, fontsize=9, style='italic')
plt.tight_layout(); plt.show()


/var/folders/ll/f8_rnhhs6sbd7zfhnvg7hbsw0000gn/T/ipykernel_42913/4164996558.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🔍 Investigative Take — Business vs. Labor

Four of seven members show zero itemized labor contributions at the $500+ threshold. The workers whose benefit adequacy these members have the power to fix are not in the donor file. The employers who benefit from frozen taxable wage bases — the same bases that haven't moved since 1992 in Maryland — are.

This is not a smoking gun. It is a pattern of access. Campaign contributions do not purchase votes. They purchase meetings, calls, and the accumulated goodwill that shapes which problems feel urgent and which feel abstract.

> *"The absence of labor in the donor file is as meaningful as the presence of business. You don't have to bribe someone to overlook you. You just have to never be in the room."*


## Chart 3 — Contribution Source Mix

As a percentage of itemized totals — individual donors vs. PACs vs. business contributions. Who's in the room where it happens?

In [6]:
fig, ax = plt.subplots(figsize=(11, 6))
style_ax(ax, 'Contribution Source Mix — UI Committee Members (% of Itemized, FEC 2024)', xlabel='Percentage of Itemized Contributions')
for i, p in enumerate(profiles):
    total = p.get('individual_contributions',0)+p.get('pac_contributions',0)+p.get('business_contributions',0)
    if total>0:
        ip = p.get('individual_contributions',0)/total*100
        pp = p.get('pac_contributions',0)/total*100
        bp = p.get('business_contributions',0)/total*100
    else: ip=pp=bp=0
    ax.barh(i, ip, color=GREEN, alpha=0.9, height=0.6, label='Individual' if i==0 else '', edgecolor=GRID, linewidth=0.4)
    ax.barh(i, pp, left=ip, color=ORANGE, alpha=0.9, height=0.6, label='PAC / Committee' if i==0 else '', edgecolor=GRID, linewidth=0.4)
    ax.barh(i, bp, left=ip+pp, color=CRIMSON, alpha=0.9, height=0.6, label='Business / Industry' if i==0 else '', edgecolor=GRID, linewidth=0.4)
names_13 = [f"{p['name']}  ({p['state']})" for p in profiles]
ax.set_yticks(range(len(profiles))); ax.set_yticklabels(names_13, color=FG, fontsize=9, fontfamily='monospace')
ax.set_xlim(0,110); ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=FG, fontsize=9, loc='lower right')
plt.tight_layout(); plt.show()

/var/folders/ll/f8_rnhhs6sbd7zfhnvg7hbsw0000gn/T/ipykernel_42913/2277067489.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🔍 Investigative Take — Contribution Mix

Individual donors fund campaigns that win elections. Business donors fund access. PAC money funds committees. The mix tells you whose calls get returned in the off-season — when the budget negotiations happen, when the legislative language gets drafted, when UI benefit caps get set and left unchanged for twelve years.

The methodology here is approximate. Keyword pattern matching on contributor names is a first pass, not a forensic verdict. But the directional signal is consistent: in this seven-member cohort, business-coded money materially exceeds labor-coded money in every case where both are present.

> *"For a complete picture: use FEC official industry codes. This analysis documents the pattern. The pattern is the story."*


## Key Findings

- **7 members** analyzed across UI-relevant committees (Ways & Means, Finance, Labor, Budget, Appropriations)
- **$89.9M ($26.1M ex-Trone)** in total 2024 cycle campaign receipts reviewed
- Business contributions dominate where labor is absent — Trone, Warner, and Hoyer each show $1M+ in business-linked itemized funding
- Ben Cline (VA), Dutch Ruppersberger (MD), and Steny Hoyer (MD) show the clearest business-to-labor funding ratios

**This analysis does not prove causation.** It documents a pattern. You can draw your own conclusions about why Maryland's taxable wage base hasn't moved since 1992.

### Honest Limitations
- Contribution categorization uses **keyword matching** — approximate, not official FEC industry codes
- Self-funding detection uses last-name matching (Trone's $63.8M is largely self-funded)
- OpenSecrets API blocked by Cloudflare — industry codes unavailable without manual registration
- Committee assignments reflect 119th Congress (2025–2027) and change each Congress

---
*The Data Vigilante · Sierra Napier, MPA · https://thedatavigilante.github.io/UI_INDEX/*

---
## ⚖️ The Conflict of Interest Matrix

| Member | Chamber | Committee (UI-relevant) | 2024 Candidate? | Business $ | Labor $ | B:L |
|--------|---------|------------------------|-----------------|------------|---------|-----|
| David Trone (MD) | House | Ways and Means, Budget | ✅ Senate race | $0 itemized | $0 | ∞ self-funded |
| Tim Kaine (VA) | Senate | Budget, HELP | ✅ Re-elected | $0 | $0 | ∞ |
| Steny Hoyer (MD) | House | Appropriations | ✅ Re-elected | $31,669 | $0 | ∞ |
| Ben Cline (VA) | House | Appropriations, Budget | ✅ Re-elected | $65,000 | $5,000 | 13:1 |
| Dutch Ruppersberger (MD) | House | Appropriations | ✅ Re-elected | $105,000 | $5,000 | 21:1 |
| Mark Warner (VA) | Senate | Finance, Banking, Budget | ⚠️ 2026 race | $0 | $0 | ∞ off-cycle |
| Chris Van Hollen (MD) | Senate | Appropriations, Banking, Budget | ⚠️ ~2028 race | $57,899 | $0 | ∞ off-cycle |

**Total 2024 candidates:** 5 of 7 (Warner + Van Hollen were not on the 2024 ballot)  
**Members with zero labor funding:** 4 of 7  
**Trone:** Co-founder Total Wine & More (~$2.4B). Ways and Means jurisdiction includes alcohol excise tax. Campaign: 98.7% self-loans.

> *Data source: fec_funding_profiles.json + political_layer_report.json. Constituent median incomes from Census ACS 2022 5-Year.*
